In [ ]:
# Standalone: cuGraph graph metrics vs T-Web eigenvalues/CWEB (Abacus CutSky)
# Run this notebook top-to-bottom; no dependencies on other notebooks.

from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except Exception as e:
    raise RuntimeError('This notebook requires pandas.') from e

try:
    import fitsio
except Exception as e:
    raise RuntimeError('This notebook requires fitsio.') from e

# --- User config ---
GRAPH_DIR = Path('/pscratch/sd/d/dkololgi/abacus/graph_constructions')
PREFIX = 'abacus_mock_alpha_23032026_cugraph'  # outputs from abacus_graph_features_cugraph.py

# Annotated catalog containing LAMBDA1/2/3 (and optionally CWEB) for the full CutSky rows.
CATALOG_PATH = Path(
    '/pscratch/sd/d/dkololgi/abacus/mocks_with_eigs/'
    'cutsky_BGS_z0.200_AbacusSummit_base_c000_ph000_with_tweb_eigs_ng2048_rs4_v2.fits'
)

# Targets
USE_CWEB_LABELS = False  # False: regress vs (LAMBDA1,2,3); True: classify vs CWEB

# Limits for speed
MAX_ROWS = 1_000_000      # None for full (can be heavy)
MI_MAX_ROWS = 300_000     # MI is slower
SEED = 42

# Output
OUT_DIR = Path('/pscratch/sd/d/dkololgi/abacus/alignment_diagnostics') / (
    'graph_corr_standalone_' + datetime.now().strftime('%Y%m%d_%H%M%S')
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'OUT_DIR: {OUT_DIR}')

NODE_FEATURES = GRAPH_DIR / f'{PREFIX}_node_features.parquet'
if not NODE_FEATURES.exists():
    raise FileNotFoundError(f'Missing node parquet: {NODE_FEATURES}')
if not CATALOG_PATH.exists():
    raise FileNotFoundError(f'Missing FITS catalog: {CATALOG_PATH}')

FEATURE_COLUMNS = ['Degree', 'Clustering', 'Density', 'Neigh Density', 'I_eig1', 'I_eig2', 'I_eig3']


def _resolve_col(dtype_names, candidates):
    names = {n.upper(): n for n in dtype_names}
    for c in candidates:
        k = c.upper()
        if k in names:
            return names[k]
    raise KeyError(f'None of candidates {list(candidates)} found. Sample: {list(dtype_names)[:20]}')


def build_graph_row_mask(table: np.ndarray) -> np.ndarray:
    """Mimic build_abacus_graph.py catalog selection: (Y1|Y5) & (BOX_INDEX != -1)."""
    names = {n.upper(): n for n in table.dtype.names}
    in_y1 = names.get('IN_Y1') or names.get('Y1')
    in_y5 = names.get('IN_Y5') or names.get('Y5')
    box_i = names.get('BOX_INDEX') or names.get('BOXINDEX')

    mask = np.ones(len(table), dtype=bool)

    if in_y1 is not None or in_y5 is not None:
        m = np.zeros(len(table), dtype=bool)
        if in_y1 is not None:
            m |= (np.asarray(table[in_y1]) == 1)
        if in_y5 is not None:
            m |= (np.asarray(table[in_y5]) == 1)
        if not m.any():
            m[:] = True
        mask &= m
    else:
        print('No IN_Y1/IN_Y5 (or Y1/Y5) columns; skipping Y1|Y5 mask.')

    if box_i is None:
        raise KeyError('BOX_INDEX column missing; cannot align to boxed graph subset.')
    mask &= (np.asarray(table[box_i]) != -1)

    return mask


print(f'Loading node features: {NODE_FEATURES}')
node_df = pd.read_parquet(NODE_FEATURES, columns=FEATURE_COLUMNS)
x = node_df.to_numpy(dtype=np.float64)
print(f'  x shape: {x.shape} (features)')

print(f'Loading targets FITS: {CATALOG_PATH}')
tab = fitsio.read(str(CATALOG_PATH))
mask = build_graph_row_mask(tab)
print(f'  Graph-build row mask keeps {int(mask.sum()):,} / {len(tab):,} rows')

if USE_CWEB_LABELS:
    cweb_col = _resolve_col(tab.dtype.names, ('CWEB', 'CWEB_CLASS', 'ENV', 'TWEB'))
    y = np.asarray(tab[cweb_col], dtype=np.int64).reshape(-1, 1)
    y = y[mask].astype(np.int64)
    target_mode = 'cweb'
    target_columns = ['cweb']
    ok = (y.ravel() >= 0) & (y.ravel() <= 3)
    y = y[ok]
    x = x[ok]
    print(f'  Target: {cweb_col} | kept {x.shape[0]:,} rows with cweb in 0..3')
else:
    l1_col = _resolve_col(tab.dtype.names, ('LAMBDA1', 'L1', 'EIG1', 'LAM1', 'LAMBDA_1'))
    l2_col = _resolve_col(tab.dtype.names, ('LAMBDA2', 'L2', 'EIG2', 'LAM2', 'LAMBDA_2'))
    l3_col = _resolve_col(tab.dtype.names, ('LAMBDA3', 'L3', 'EIG3', 'LAM3', 'LAMBDA_3'))
    y = np.stack([
        np.asarray(tab[l1_col], dtype=np.float64),
        np.asarray(tab[l2_col], dtype=np.float64),
        np.asarray(tab[l3_col], dtype=np.float64),
    ], axis=1)
    y = y[mask]
    target_mode = 'eigenvalues'
    target_columns = ['lambda1', 'lambda2', 'lambda3']

    ok = np.isfinite(y).all(axis=1)
    if not ok.all():
        print(f'  Dropping {int((~ok).sum()):,} rows with non-finite eigenvalues')
        y = y[ok]
        x = x[ok]

print(f'Aligned rows: x={x.shape[0]:,} y={y.shape[0]:,}')
if x.shape[0] != y.shape[0]:
    raise ValueError(
        f'Row mismatch after masking: features={x.shape[0]:,}, targets={y.shape[0]:,}. '
        'This implies feature parquet and FITS mask do not describe the same row order.'
    )

rng = np.random.default_rng(SEED)
if MAX_ROWS is not None and x.shape[0] > int(MAX_ROWS):
    idx = rng.choice(x.shape[0], size=int(MAX_ROWS), replace=False)
    x = x[idx]
    y = y[idx]
    print(f'Using subsample for correlations: {x.shape[0]:,} rows')

# Pearson
xc = x - x.mean(axis=0, keepdims=True)
yc = y - y.mean(axis=0, keepdims=True)
num = xc.T @ yc
xnorm = np.sqrt(np.sum(xc * xc, axis=0, keepdims=True)).T
ynorm = np.sqrt(np.sum(yc * yc, axis=0, keepdims=True))
pearson = num / np.maximum(xnorm * ynorm, 1e-12)

pearson_df = pd.DataFrame(pearson, index=FEATURE_COLUMNS, columns=target_columns)
print('\nPearson correlation:')
print(pearson_df)

pearson_out = OUT_DIR / f'graph_metric_pearson_vs_{target_mode}.csv'
pearson_df.to_csv(pearson_out, index=True)
print(f'Saved: {pearson_out}')

fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
im = ax.imshow(pearson_df.values, aspect='auto', cmap='coolwarm', vmin=-1.0, vmax=1.0)
ax.set_title('Pearson: Graph metrics vs ' + ('CWEB' if USE_CWEB_LABELS else 'eigenvalues'))
ax.set_xlabel('Target')
ax.set_ylabel('Feature')
ax.set_xticks(np.arange(pearson_df.shape[1]))
ax.set_xticklabels(list(pearson_df.columns))
ax.set_yticks(np.arange(pearson_df.shape[0]))
ax.set_yticklabels(list(pearson_df.index))
for i in range(pearson_df.shape[0]):
    for j in range(pearson_df.shape[1]):
        ax.text(j, i, f'{pearson_df.values[i, j]:.3f}', ha='center', va='center', fontsize=8)
cb = fig.colorbar(im, ax=ax)
cb.set_label('Pearson r')
out = OUT_DIR / f'graph_metric_pearson_heatmap_{target_mode}.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

# MI (optional)
try:
    from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

    xm = x
    ym = y
    if MI_MAX_ROWS is not None and x.shape[0] > int(MI_MAX_ROWS):
        idx_mi = rng.choice(x.shape[0], size=int(MI_MAX_ROWS), replace=False)
        xm = x[idx_mi]
        ym = y[idx_mi]
        print(f'Using subsample for MI: {xm.shape[0]:,} rows')

    if USE_CWEB_LABELS:
        yc_flat = np.asarray(ym.ravel(), dtype=np.int64)
        mi_vec = mutual_info_classif(xm, yc_flat, discrete_features=False, discrete_target=True, random_state=SEED)
        mi_df = pd.DataFrame(mi_vec.reshape(-1, 1), index=FEATURE_COLUMNS, columns=['cweb'])
    else:
        mi = np.zeros((xm.shape[1], ym.shape[1]), dtype=np.float64)
        for j in range(ym.shape[1]):
            mi[:, j] = mutual_info_regression(xm, ym[:, j], random_state=SEED)
        mi_df = pd.DataFrame(mi, index=FEATURE_COLUMNS, columns=target_columns)

    print('\nMutual information:')
    print(mi_df)
    mi_out = OUT_DIR / f'graph_metric_mutual_info_vs_{target_mode}.csv'
    mi_df.to_csv(mi_out, index=True)
    print(f'Saved: {mi_out}')
except Exception as e:
    print(f'Skipping MI (sklearn missing or runtime error): {e}')

print('Done.')